In [7]:
import arxiv

In [8]:
# this method helps to search for papers based on the query (which can have categories, titles, authors, etc.), the maximum number of results to return, and the sorting criterion (e.g., relevance, last updated date, etc.)
search = arxiv.Search(
    query="cat:cs.AI",
    max_results=5,
    sort_by=arxiv.SortCriterion.Relevance
)

# this method creates a client that can be used to fetch the results of the search query
client = arxiv.Client()

# this method fetches the results of the search query using the client and returns an iterable of paper objects
results = client.results(search)

for paper in results:
    print(paper.title)
    print(paper.pdf_url)
    print("-" * 80)

A Deep Reinforcement Learning Approach for Ramp Metering Based on Traffic Video Data
https://arxiv.org/pdf/2012.12104v1
--------------------------------------------------------------------------------
Rethink AI-based Power Grid Control: Diving Into Algorithm Design
https://arxiv.org/pdf/2012.13026v1
--------------------------------------------------------------------------------
Fuzzy Commitments Offer Insufficient Protection to Biometric Templates Produced by Deep Learning
https://arxiv.org/pdf/2012.13293v1
--------------------------------------------------------------------------------
Generalization in portfolio-based algorithm selection
https://arxiv.org/pdf/2012.13315v1
--------------------------------------------------------------------------------
I like fish, especially dolphins: Addressing Contradictions in Dialogue Modeling
https://arxiv.org/pdf/2012.13391v2
--------------------------------------------------------------------------------


In [9]:
print("Title:", paper.title)
authors = [author.name for author in paper.authors]
print(authors)
print("Summary:", paper.summary)
print("Published:", paper.published)
print("Updated:", paper.updated)
print("PDF URL:", paper.pdf_url)
print("Entry ID:", paper.entry_id)
print("Primary category:", paper.primary_category)
print("Categories:", paper.categories)

Title: I like fish, especially dolphins: Addressing Contradictions in Dialogue Modeling
['Yixin Nie', 'Mary Williamson', 'Mohit Bansal', 'Douwe Kiela', 'Jason Weston']
Summary: To quantify how well natural language understanding models can capture consistency in a general conversation, we introduce the DialoguE COntradiction DEtection task (DECODE) and a new conversational dataset containing both human-human and human-bot contradictory dialogues. We then compare a structured utterance-based approach of using pre-trained Transformer models for contradiction detection with the typical unstructured approach. Results reveal that: (i) our newly collected dataset is notably more effective at providing supervision for the dialogue contradiction detection task than existing NLI data including those aimed to cover the dialogue domain; (ii) the structured utterance-based approach is more robust and transferable on both analysis and out-of-distribution dialogues than its unstructured counterpart.

In [10]:
import arxiv
import json
from pathlib import Path
import requests


metadata_dir = Path("../data/metadata")
metadata_dir.mkdir(parents=True, exist_ok=True)

all_metadata = []

for paper in client.results(search):
    paper_id = paper.get_short_id().replace("/", "_")
    category = paper.primary_category

    pdf_dir = Path("../data/raw/arxiv") / category
    pdf_dir.mkdir(parents=True, exist_ok=True)

    pdf_path = pdf_dir / f"{paper_id}.pdf"

    if not pdf_path.exists():
        # this method creates a get request to extract the url of the paper
        response = requests.get(paper.pdf_url)
        
        if response.status_code == 200:
            with open(pdf_path,"wb") as f:
                f.write(response.content)
            print(f"Downloaded PDF for paper: {pdf_path}")
        else:
            print(f"Failed to download PDF for paper: {paper.title}, status code: {response.status_code}")
            continue


    metadata = {
        "paper_id": paper.get_short_id(),
        "title": paper.title,
        "authors": [author.name for author in paper.authors],
        "summary": paper.summary,
        "published": paper.published.isoformat(),
        "updated": paper.updated.isoformat(),
        "primary_category": paper.primary_category,
        "categories": paper.categories,
        "pdf_url": paper.pdf_url,
        "entry_id": paper.entry_id,
        "local_pdf_path": str(pdf_path)
    }

    all_metadata.append(metadata)

metadata_path = metadata_dir / "papers.json"

with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(all_metadata, f, indent=2, ensure_ascii=False)

print(f"Saved {len(all_metadata)} papers")
print(f"Metadata saved to: {metadata_path}")

Saved 5 papers
Metadata saved to: ..\data\metadata\papers.json


In [11]:
import fitz

In [12]:
# pdf_path = Path(r"..\data\raw\arxiv\cs.AI\2012.13026v1.pdf")

# doc = fitz.open(pdf_path)

# print("Number of pages:", doc.page_count)

# page = doc[0]

# text = page.get_text("text")

# print(text[:1000])  # Print the first 1000 characters of the text

In [13]:
pdf_path = Path(r"..\data\raw\arxiv\cs.AI\2012.13026v1.pdf")

doc = fitz.open(pdf_path)

pages = []

for page_number, page in enumerate(doc, start=1):
    text = page.get_text("text")

    pages.append({
        "page_number": page_number,
        "text": text
    })

doc.close()

print("Extracted pages:", len(pages))
print(pages)

Extracted pages: 8
[{'page_number': 1, 'text': 'Rethink AI-based Power Grid Control: Diving Into\nAlgorithm Design\nXiren Zhou1, Siqi Wang2, Ruisheng Diao2, Desong Bian2, Jiajun Duan2 and Di Shi2\n1Columbia University\n1 xz2754@columbia.edu\n2Global Energy Interconnection Research Institute North America(GEIRINA)\n2{siqi.wang, ruisheng.diao, desong.bian, jiajun.duan, di.shi}@geirina.net\nAbstract\nRecently, deep reinforcement learning (DRL)-based approach has shown promise\nin solving complex decision and control problems in power engineering domain.\nIn this paper, we present an in-depth analysis of DRL-based voltage control from\naspects of algorithm selection, state space representation, and reward engineering.\nTo resolve observed issues, we propose a novel imitation learning-based approach\nto directly map power grid operating points to effective actions without any interim\nreinforcement learning process. The performance results demonstrate that the\nproposed approach has strong 

In [14]:
from dataclasses import dataclass
from pathlib import Path
from typing import List, Dict, Any
import fitz

@dataclass
class RawPage:
    page_number: int
    text: str
    char_count: int

@dataclass
class RawDocument:
    paper_id: str
    pdf_path: str
    pages_count: int
    pages: List[RawPage]

In [15]:
def extract_pdf_text(pdf_path: str | Path, paper_id: str | None = None) -> RawDocument:
    pdf_path = Path(pdf_path)

    if not pdf_path.exists():
        raise FileNotFoundError(f"PDF file not found: {pdf_path}")

    if paper_id is None:
        paper_id = pdf_path.stem

    pages = []

    with fitz.open(pdf_path) as doc:
        for page_number, page in enumerate(doc, start=1):
            text = page.get_text("text")
            char_count = len(text)

            pages.append(RawPage(
                page_number=page_number,
                text=text,
                char_count=char_count
            ))
        
        raw_doc = RawDocument(
            paper_id=paper_id,
            pdf_path=str(pdf_path),
            pages_count=doc.page_count,
            pages=pages
        )

    return raw_doc

In [16]:
raw_doc = extract_pdf_text(r"..\data\raw\arxiv\cs.AI\2012.13026v1.pdf")
print(raw_doc.paper_id)
print(raw_doc.pages_count)
print(raw_doc.pages[0].text[:1000])

2012.13026v1
8
Rethink AI-based Power Grid Control: Diving Into
Algorithm Design
Xiren Zhou1, Siqi Wang2, Ruisheng Diao2, Desong Bian2, Jiajun Duan2 and Di Shi2
1Columbia University
1 xz2754@columbia.edu
2Global Energy Interconnection Research Institute North America(GEIRINA)
2{siqi.wang, ruisheng.diao, desong.bian, jiajun.duan, di.shi}@geirina.net
Abstract
Recently, deep reinforcement learning (DRL)-based approach has shown promise
in solving complex decision and control problems in power engineering domain.
In this paper, we present an in-depth analysis of DRL-based voltage control from
aspects of algorithm selection, state space representation, and reward engineering.
To resolve observed issues, we propose a novel imitation learning-based approach
to directly map power grid operating points to effective actions without any interim
reinforcement learning process. The performance results demonstrate that the
proposed approach has strong generalization ability with much less training t

In [17]:
import json
from dataclasses import asdict
from pathlib import Path


def save_raw_document(raw_doc: RawDocument, output_dir: str | Path):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    output_path = output_dir / f"{raw_doc.paper_id}.json"

    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(asdict(raw_doc), f, ensure_ascii=False, indent=2)

    return output_path

In [18]:
output_path = save_raw_document(
    raw_doc,
    output_dir="../data/processed/raw_text"
)

print("Saved to:", output_path)

Saved to: ..\data\processed\raw_text\2012.13026v1.json


In [19]:
RAW_DIR = Path("../data/raw/arxiv")
OUTPUT_DIR = Path("../data/processed/raw_text")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


for category_folder in RAW_DIR.iterdir():

    # skip non-folders (safety check)
    if not category_folder.is_dir():
        continue

    print(f"\n📁 Processing category: {category_folder.name}")

    # loop through PDFs in category
    for pdf_file in category_folder.glob("*.pdf"):

        try:
            print(f"   📄 Extracting: {pdf_file.name}")

            # 1. extract text page-by-page
            raw_doc = extract_pdf_text(pdf_file)

            # 2. build output path (keep category structure)
            category_output_dir = OUTPUT_DIR / category_folder.name
            category_output_dir.mkdir(parents=True, exist_ok=True)

            output_path = category_output_dir / f"{raw_doc.paper_id}.json"

            # 3. save JSON
            with open(output_path, "w", encoding="utf-8") as f:
                json.dump(asdict(raw_doc), f, ensure_ascii=False, indent=2)

        except Exception as e:
            print(f"   ❌ Failed {pdf_file.name}: {e}")


📁 Processing category: cs.AI
   📄 Extracting: 2012.13026v1.pdf
   📄 Extracting: 2012.13315v1.pdf

📁 Processing category: cs.CL
   📄 Extracting: 2012.13391v2.pdf

📁 Processing category: cs.CR
   📄 Extracting: 2012.13293v1.pdf

📁 Processing category: cs.CV
   📄 Extracting: 2012.12104v1.pdf

📁 Processing category: cs.LG


In [20]:
import re
from dataclasses import dataclass
from typing import Dict, List

In [21]:
@dataclass
class SectionedDoc:
    paper_id: str
    sections: Dict[str,str]

In [33]:
SECTION_PATTERNS = {
    "abstract": r"^(\d+\.?\s*)?(abstract|summary)\s*$",
    "introduction": r"^(\d+\.?\s*)?(introduction)\s*$",
    "related_work": r"^(\d+\.?\s*)?(related work|related works|background|prior work)\s*$",
    "method": r"^(\d+\.?\s*)?(methodology|methods|method|approach|proposed method|model)\s*$",
    "experiments": r"^(\d+\.?\s*)?(experiments|experimental setup|evaluation|results|experiments and results)\s*$",
    "conclusion": r"^(\d+\.?\s*)?(conclusion|conclusions|discussion and conclusion)\s*$",
}

In [34]:


def detect_section(line: str):
    line_clean = line.strip().lower()
    line_clean = re.sub(r"\s+", " ", line_clean)

    # Ignore empty lines
    if not line_clean:
        return None

    # Important:
    # If the line is long, it is probably a sentence, not a heading.
    if len(line_clean.split()) > 7:
        return None

    # Check if the line ends with a period, which is unusual for a heading
    if line_clean.endswith("."):
        return None

    # Check if the line contains common sentence starters
    sentence_starters = [
        "we ", "our ", "this ", "that ", "the ", "a ", "an ", "in ", "on ", "for ",
        "with ", "by ", "from ", "to ", "of ", "is ", "are ", "was ", "were ", "has ",
        "have ", "had ", "do ", "does ", "did ", "can ", "could ", "will ", "would ",
        "shall ", "should ", "may ", "might ", "must "
    ]
    
    for starter in sentence_starters:
        if line_clean.startswith(starter):
            return None

    # Check for common verbs that indicate a sentence rather than a heading
    common_verbs = [
        "uses ", "used ", "using ", "propose ", "proposes ", "proposed ", 
        "implement ", "implements ", "implemented ", "show ", "shows ", "showed ",
        "demonstrate ", "demonstrates ", "demonstrated ", "present ", "presents ",
        "presented ", "introduce ", "introduces ", "introduced ", "describe ",
        "describes ", "described ", "explain ", "explains ", "explained "
    ]
    
    for verb in common_verbs:
        if line_clean.startswith(verb):
            return None

    for section, pattern in SECTION_PATTERNS.items():
        if re.match(pattern, line_clean):
            return section

    return None


In [35]:
def extract_sections(pages, paper_id):
    sections = {
        "abstract": "",
        "introduction": "",
        "related_work": "",
        "method": "",
        "experiments": "",
        "conclusion": ""
    }

    current_section = None

    for page in pages:
        lines = page.text.split("\n")

        for line in lines:
            detected = detect_section(line)

            if detected:
                current_section = detected
                continue

            if current_section:
                sections[current_section] += line + " "

    for k in sections:
        sections[k] = sections[k].strip()

    return SectionedDoc(
        paper_id=paper_id,
        sections=sections
    )

In [36]:
sectioned_doc = extract_sections(
    pages=raw_doc.pages,
    paper_id="test_paper"
)

print(sectioned_doc.sections.keys())

dict_keys(['abstract', 'introduction', 'related_work', 'method', 'experiments', 'conclusion'])


In [ ]:
print(sectioned_doc.sections["abstract"])

Ramp metering that uses traffic signals to regulate vehicle flows from the on-ramps has been  widely implemented to improve vehicle mobility of the freeway. Previous studies generally  update signal timings in real-time based on predefined traffic measures collected by point  detectors, such as traffic volumes and occupancies. Comparing with point detectors, traffic  cameras—which have been increasingly deployed on road networks—could cover larger areas  and provide more detailed traffic information. In this work, we propose a deep reinforcement


In [ ]:
# with open("abstract_test.txt", "w", encoding="utf-8") as f:
#     f.write(sectioned_doc.sections["related_work"])